In [1]:
#Este intento fue uno de los primeros el cual decidimos entrenar una regresion logistica unicamente con solicitudes train, lo cual no funciono y nos arrogo un resultado negativo(-5000), esto se debe a que el modelo
#no cuenta con informacion del comportamiento de los clientes.

In [2]:
!pip install sklearn-pandas pandas scikit-learn


[notice] A new release of pip is available: 23.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn_pandas import DataFrameMapper

URL_BASE = "https://raw.githubusercontent.com/FranciscoAmbort/grupo13_castellano_ambort_ia_2026/main/entrega_3/data"

url_train = f"{URL_BASE}/solicitudes_train.csv"
url_test = f"{URL_BASE}/solicitudes_test.csv"
url_comp = f"{URL_BASE}/comportamiento_bancario.csv"

# 2. Carga directa optimizada
df_sol_train = pd.read_csv(url_train)
df_sol_test = pd.read_csv(url_test)
comportamiento = pd.read_csv(url_comp)


# Regresion Logistica con solicitudes train


In [4]:
from sklearn.model_selection import train_test_split

In [5]:
train,not_train= train_test_split(df_sol_train,test_size=0.4,random_state=42)
validation,test= train_test_split(not_train,test_size=0.5,random_state=42)

In [6]:
sample=train.sample(5,random_state=42)
sample

,ID_Solicitud,ID_Usuario,Monto_Solicitado,Tasa_Interes,Ingreso_Mensual,Motivo_Prestamo,Fecha_Solicitud,Estado_Final
19978,19979,5199,16800,0.3160,2926,Negocio,2025-05,0
14882,14883,12815,10700,0.3018,2073,Consumo,2025-10,1
3907,3908,4858,5000,0.2390,2445,Consumo,2025-08,1
9709,9710,12769,37500,0.2806,5064,Vivienda,2025-01,0
18215,18216,6646,4100,0.2506,1278,Vehiculo,2025-06,1


In [7]:
mapper_baseline = DataFrameMapper([
    (['Monto_Solicitado'], [StandardScaler()]),
    (['Tasa_Interes'], [StandardScaler()]),
    (['Ingreso_Mensual'], [StandardScaler()]),
    (['Motivo_Prestamo'], [OneHotEncoder()]),
])

mapper_baseline.fit(train)
mapper_baseline.transform(sample)

c:\Users\Matias\Desktop\intentos\env\Lib\site-packages\sklearn\utils\deprecation.py:87: FutureWarning: Function tosequence is deprecated; tosequence was deprecated in 1.5 and will be removed in 1.7
  warnings.warn(msg, category=FutureWarning)


array([[ 0.19517476,  1.75134974, -0.51445368,  0.        ,  0.        ,
         1.        ,  0.        ,  0.        ,  0.        ,  0.        ],
       [-0.27219025,  1.31218723, -0.85590322,  1.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ],
       [-0.70890838, -0.63002444, -0.7069944 ,  1.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ],
       [ 1.78115112,  0.65653616,  0.34137179,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  1.        ],
       [-0.77786387, -0.27127197, -1.1741358 ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  1.        ,  0.        ]])

In [8]:
mapper_baseline.transformed_names_

['Monto_Solicitado',
 'Tasa_Interes',
 'Ingreso_Mensual',
 'Motivo_Prestamo_0',
 'Motivo_Prestamo_1',
 'Motivo_Prestamo_2',
 'Motivo_Prestamo_3',
 'Motivo_Prestamo_4',
 'Motivo_Prestamo_5',
 'Motivo_Prestamo_6']

In [9]:
lr_model_si = Pipeline([
    ('mapper', mapper_baseline),
    ('classifier', LogisticRegression(random_state=42)),
])

lr_model_si.fit(train, train.Estado_Final)

y_pred = lr_model_si.predict(validation)


c:\Users\Matias\Desktop\intentos\env\Lib\site-packages\sklearn\utils\deprecation.py:87: FutureWarning: Function tosequence is deprecated; tosequence was deprecated in 1.5 and will be removed in 1.7
  warnings.warn(msg, category=FutureWarning)
c:\Users\Matias\Desktop\intentos\env\Lib\site-packages\sklearn\linear_model\_logistic.py:455: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(


In [10]:
from sklearn import metrics

In [11]:
metrics.accuracy_score(validation.Estado_Final, y_pred)

0.8412771864877372

In [12]:

pipeline_lr = Pipeline([
    ('mapper', mapper_baseline),
    ('classifier', LogisticRegression(random_state=42))
])
pipeline_lr.fit(df_sol_train, df_sol_train['Estado_Final'])

y_test_pred = pipeline_lr.predict(df_sol_test)

c:\Users\Matias\Desktop\intentos\env\Lib\site-packages\sklearn\utils\deprecation.py:87: FutureWarning: Function tosequence is deprecated; tosequence was deprecated in 1.5 and will be removed in 1.7
  warnings.warn(msg, category=FutureWarning)
c:\Users\Matias\Desktop\intentos\env\Lib\site-packages\sklearn\linear_model\_logistic.py:455: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(


In [13]:
test_eval = df_sol_test.copy()
test_eval['pred'] = y_test_pred

candidatos = test_eval[test_eval['pred'] == 1].sort_values(
    by='Monto_Solicitado',
    ascending=False
)

PRESUPUESTO_MAX = 2_000_000
monto_acumulado = 0
ids_aprobados = []

for idx, fila in candidatos.iterrows():
    monto = fila['Monto_Solicitado']
    id_solicitud = fila['ID_Solicitud']

    if monto_acumulado + monto <= PRESUPUESTO_MAX:
        monto_acumulado += monto
        ids_aprobados.append(id_solicitud)
    else:
        continue

test_eval['Estado_Final'] = test_eval['ID_Solicitud'].isin(ids_aprobados).astype(int)

print(f"Solicitudes aprobadas: {len(ids_aprobados)}")
print(f"Monto total asignado: ${monto_acumulado:,.2f} USD")

Solicitudes aprobadas: 267
Monto total asignado: $2,000,000.00 USD


In [14]:
#entrega
submission = pd.DataFrame({
    'ID_Solicitud': test_eval['ID_Solicitud'],
    'Decision_Aprobacion': test_eval['Estado_Final']
})

submission.to_csv('submission_segundo_intento.csv', index=False)
print("OK")

OK
